# 46 — Pre-production evidence

Synthesizes M5 evidence into a robustness card and monitoring contract. Passing this notebook **does not authorize production**.


In [ ]:
from argos.common.paths import ProjectPaths
from argos.config.gold import load_gold_spec
from argos.config.modeling import load_modeling_spec
from argos.config.robustness import load_robustness_spec

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()
(paths.root / "reports" / "robustness").mkdir(parents=True, exist_ok=True)
(paths.root / "data" / "m5").mkdir(parents=True, exist_ok=True)
(paths.root / "artifacts" / "models" / "m5").mkdir(parents=True, exist_ok=True)

m4_gate = paths.docs / "decisions" / "M4_GATE.md"
assert m4_gate.exists() and "M4 ✅ CLOSED" in m4_gate.read_text(encoding="utf-8"), (
    "M4 no está formalmente cerrado. No iniciar M5."
)
GOLD_SPEC = load_gold_spec(paths.configs / "gold.toml")
MODEL_SPEC = load_modeling_spec(paths.configs / "modeling.toml")
M5_SPEC = load_robustness_spec(paths.configs / "robustness.toml")
MANIFEST_DIR = paths.root / "artifacts" / "manifests"

assert GOLD_SPEC.schema_version == M5_SPEC.core.benchmark_gold_schema
assert MODEL_SPEC.schema_version == M5_SPEC.core.benchmark_model_schema
{
    "gold_schema": GOLD_SPEC.schema_version,
    "model_schema": MODEL_SPEC.schema_version,
    "m5_schema": M5_SPEC.schema_version,
    "benchmark_model": M5_SPEC.core.benchmark_model_name,
    "benchmark_calibration": M5_SPEC.core.benchmark_calibration,
    "consumed_test_year": M5_SPEC.governance.consumed_test_year,
    "fresh_oot_year": M5_SPEC.governance.fresh_oot_year,
}

import pandas as pd
path45 = paths.reports / "robustness" / "45_validation.parquet"
assert path45.exists(), "Ejecuta 45 primero."
check45 = pd.read_parquet(path45)
assert check45.loc[check45["blocking"], "passed"].all(), "45 no está aprobado."
from argos.robustness.readiness import BUILD_IMPLEMENTATION
BUILD_IMPLEMENTATION


In [ ]:
from argos.robustness.readiness import build_preproduction_evidence

result = build_preproduction_evidence(
    root=paths.root,
    reports_root=paths.reports,
    manifest_dir=MANIFEST_DIR,
    docs_root=paths.docs,
    modeling_spec=MODEL_SPEC,
    robustness_spec=M5_SPEC,
)
result.evidence_summary


In [ ]:
result.monitoring_thresholds


In [ ]:
result.validation


In [ ]:
blocking = result.validation.loc[result.validation["blocking"]]
print("46 ✅ PASS" if result.ready_for_next_notebook else "46 ❌ FAIL")
print(f"Blocking checks: {int(blocking['passed'].sum())}/{len(blocking)}")
print("Robustness card:", result.robustness_card_path)
print("Monitoring spec:", result.monitoring_spec_path)
print("Manifest:", result.manifest_path)
assert result.ready_for_next_notebook
